# 01 · Data audit, signals and views

From the raw Datastream file to the files the backtest and the app use: clean prices and returns, a data quality report, the two signals and the monthly views. Nothing is interpolated or filled: every problem found is logged with its decision in `data/reports/data_quality_report.csv`.

Needs `data/raw/ETF.xlsx` (licensed Datastream export, not in the repository).

In [1]:
# Setup: find the project root, so the notebook runs from any folder
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").exists())
sys.path.insert(0, str(ROOT))
from src import data

ETF_FILE, RF_FILE = data.ETF_FILE, data.RF_FILE                 # data/raw/ (ETF.xlsx is not in the repository)
PROCESSED, REPORTS = data.PROCESSED_DIR, data.REPORTS_DIR

## 1. Raw data

Two sheets (price and total return index), one column per ETF, identified by a keyword in its Datastream name.

In [2]:
import pandas as pd

xls = pd.ExcelFile(ETF_FILE)
print("Sheets in the file:", xls.sheet_names)

Sheets in the file: ['Price', 'Total return index', 'interest rate ']


In [3]:
pd.set_option("display.max_columns", 20)

for sheet in ["Price", "Total return index"]:
    raw = pd.read_excel(ETF_FILE, sheet_name=sheet, header=None)
    print(f"--- {sheet}: {raw.shape[0]} rows, {raw.shape[1]} columns ---")
    display(raw.head(6))
    display(raw.tail(3))

--- Price: 5179 rows, 9 columns ---


,0,1,2,3,4,5,6,7,8
0,Name,ISHARES MSCI EMERGING MARKETS ETF,ISHARES MSCI EAFE,STATE STREET SPDR S&P 500 ETF TRUST,ISHARES 7-10 YR.TRSY.BD.,ISHARES IBOXX $ INV GRADE CORPORATE BOND ETF,ISHARES 20+ YEAR TREASURY BOND ETF,VANGUARD REAL ESTATE ETF,SPDR GOLD SHARES
1,2007-01-01 00:00:00,38.0566,73.22,141.62,82.44,106.68,88.43,77,63.21
2,2007-01-02 00:00:00,38.0566,73.22,141.62,82.44,106.68,88.43,77,63.21
3,2007-01-03 00:00:00,38.38,73.51,141.37,82.68,106.45,89.07,77.12,62.28
4,2007-01-04 00:00:00,37.85,73.32,141.67,82.95,107.25,89.6,77.22,61.65
5,2007-01-05 00:00:00,36.7433,72.2,140.54,82.71,107.19,89.22,75.79,60.17


,0,1,2,3,4,5,6,7,8
5176,2026-11-02 00:00:00,61.11,103.7,752.1001,91.5,103.71,81.73,95.06,361.6499
5177,2026-11-03 00:00:00,61.1,103.08,754.3501,91.16,103.72,81.11,95.01,359.26
5178,2026-11-04 00:00:00,61.62,105.72,766.8101,91.66,104.5,81.76,98.03,372.49


--- Total return index: 5179 rows, 9 columns ---


,0,1,2,3,4,5,6,7,8
0,Name,STATE STREET SPDR S&P 500 ETF TRUST - TOT RETU...,ISHARES MSCI EAFE - TOT RETURN IND,ISHARES MSCI EMERGING MARKETS ETF - TOT RETURN...,ISHARES 7-10 YR.TRSY.BD. - TOT RETURN IND,ISHARES IBOXX $ INV GRADE CORPORATE BOND ETF -...,ISHARES 20+ YEAR TREASURY BOND ETF - TOT RETUR...,VANGUARD REAL ESTATE ETF - TOT RETURN IND,SPDR GOLD SHARES - TOT RETURN IND
1,2007-01-01 00:00:00,410.44,190.23,358.35,118.85,129.76,132.61,175.65,142.43
2,2007-01-02 00:00:00,410.44,190.23,358.35,118.85,129.76,132.61,175.65,142.43
3,2007-01-03 00:00:00,409.72,190.99,361.39,119.2,129.48,133.57,175.93,140.33
4,2007-01-04 00:00:00,410.59,190.49,356.4,119.59,130.45,134.37,176.15,138.91
5,2007-01-05 00:00:00,407.31,187.58,345.98,119.24,130.38,133.8,172.89,135.58


,0,1,2,3,4,5,6,7,8
5176,2026-11-02 00:00:00,3126.42,482.12,858.94,219.81,277.72,229.5,490.59,814.89
5177,2026-11-03 00:00:00,3135.78,479.24,858.8,218.99,277.75,227.75,490.33,809.51
5178,2026-11-04 00:00:00,3187.57,491.51,866.11,220.19,279.84,229.58,505.92,839.32


In [4]:
TICKER_KEYWORDS = {
    "SPY": "S&P 500",
    "EFA": "EAFE",
    "EEM": "EMERGING",
    "IEF": "7-10",
    "LQD": "INV GRADE",
    "TLT": "20+",
    "VNQ": "REAL ESTATE",
    "GLD": "GOLD",
}


def read_datastream_sheet(sheet):
    df = pd.read_excel(ETF_FILE, sheet_name=sheet, header=0)
    df = df.rename(columns={df.columns[0]: "Date"})

    # Check the dates
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    print(f"\n--- {sheet} ---")
    print("Invalid dates:", df["Date"].isna().sum())
    print("Duplicated dates:", df["Date"].dropna().duplicated().sum())
    df = df.dropna(subset=["Date"])

    # Identify each ETF by a keyword in its Datastream name
    mapping = {}
    for col in df.columns[1:]:
        matches = [t for t, kw in TICKER_KEYWORDS.items() if kw in str(col).upper()]
        assert len(matches) == 1, f"Ambiguous or unknown column: {col}"
        mapping[col] = matches[0]
    assert len(mapping) == 8
    assert set(mapping.values()) == set(TICKER_KEYWORDS)
    df = df.rename(columns=mapping)

    # Check non-numeric values  
    for ticker in TICKER_KEYWORDS:
        numeric = pd.to_numeric(df[ticker], errors="coerce")
        invalid = df[ticker].notna() & numeric.isna()
        print(f"{ticker}: {invalid.sum()} non-numeric values")
        df[ticker] = numeric

    return df.set_index("Date").sort_index()[list(TICKER_KEYWORDS)]


price = read_datastream_sheet("Price")
ri = read_datastream_sheet("Total return index")

print()
for name, df in [("Price", price), ("RI", ri)]:
    print(f"{name}: {df.shape[0]} dates, from {df.index.min().date()} to {df.index.max().date()}")
    print("  Columns:", list(df.columns))
print("Same dates in both sheets:", price.index.equals(ri.index))


--- Price ---
Invalid dates: 0
Duplicated dates: 0
SPY: 0 non-numeric values
EFA: 0 non-numeric values
EEM: 0 non-numeric values
IEF: 0 non-numeric values
LQD: 0 non-numeric values
TLT: 0 non-numeric values
VNQ: 0 non-numeric values
GLD: 0 non-numeric values

--- Total return index ---
Invalid dates: 0
Duplicated dates: 0
SPY: 0 non-numeric values
EFA: 0 non-numeric values
EEM: 0 non-numeric values
IEF: 0 non-numeric values
LQD: 0 non-numeric values
TLT: 0 non-numeric values
VNQ: 0 non-numeric values
GLD: 0 non-numeric values

Price: 5178 dates, from 2007-01-01 to 2026-11-04
  Columns: ['SPY', 'EFA', 'EEM', 'IEF', 'LQD', 'TLT', 'VNQ', 'GLD']
RI: 5178 dates, from 2007-01-01 to 2026-11-04
  Columns: ['SPY', 'EFA', 'EEM', 'IEF', 'LQD', 'TLT', 'VNQ', 'GLD']
Same dates in both sheets: True


## 2. Audit and cleaning

Only official NYSE sessions are kept: holiday rows copied from the previous day would create fake zero returns and understate volatility. Missing values, non-positive prices, very large moves and stale series are checked; each case is logged as kept or excluded, with its reason.

In [ ]:
import pandas_market_calendars as mcal

# Data quality report: one row per check (what was detected, kept or excluded, and why),
# saved to data/reports/ so every cleaning decision can be traced.
report = []

def add_check(check, ticker, dates, detected, kept=0, excluded=0, corrected=0, justification=""):
    report.append({
        "Check": check,
        "Ticker": ticker,
        "Dates": ", ".join(str(d.date()) for d in dates[:10]) + (" ..." if len(dates) > 10 else ""),
        "Detected": detected,
        "Kept": kept,
        "Excluded": excluded,
        "Corrected": corrected,
        "Justification": justification,
    })

# 1. NYSE calendar: keep only official trading days; holiday rows copied from the day before would create fake zero returns and understate volatility.
nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(start_date=ri.index.min(), end_date=ri.index.max())
sessions = pd.DatetimeIndex(schedule.index).normalize()

file_dates = ri.index.normalize()
non_sessions = file_dates.difference(sessions)      # in the file, market closed
missing_sessions = sessions.difference(file_dates)  # market open, missing from the file

prev = ri.shift(1).loc[non_sessions]
copies = (ri.loc[non_sessions] == prev).all(axis=1)
n_first = int(len(non_sessions) > 0 and non_sessions[0] == ri.index.min())

print(f"NYSE sessions: {len(sessions)}")
print(f"Non-NYSE dates in file: {len(non_sessions)} (exact copies of previous day: {copies.sum()})")
print(f"Missing NYSE sessions: {len(missing_sessions)}")

add_check("Non-NYSE date", "All", non_sessions, len(non_sessions), excluded=len(non_sessions),
          justification=(f"NYSE closed; {copies.sum()}/{len(non_sessions)} dates match the previous "
                         f"observation; {n_first} first date(s) without predecessor"))
add_check("Missing NYSE session", "All", missing_sessions, len(missing_sessions),
          justification="No case" if len(missing_sessions) == 0 else "Common date excluded")

# Keep NYSE sessions only (raw data stays untouched)
price_clean = price.loc[price.index.isin(sessions)].copy()
ri_clean = ri.loc[ri.index.isin(sessions)].copy()

assert price_clean.index.equals(ri_clean.index), "Price and RI dates are misaligned"
assert ri_clean.index.equals(sessions), (
    "Some NYSE sessions are missing: apply the V1.4 rule (exclude the common date, flag irregular returns)"
)
print(f"After filtering: {len(ri_clean)} sessions, {ri_clean.index.min().date()} to {ri_clean.index.max().date()}")

# 2. Missing values: stop if any price is still missing after cleaning, then compute returns without filling any gap.
assert not ri_clean.isna().any().any(), "Missing values in RI: review needed"
assert not price_clean.isna().any().any(), "Missing values in Price: review needed"

r_ri = ri_clean.pct_change(fill_method=None)
r_p = price_clean.pct_change(fill_method=None)

# 3. Value checks: look for non-positive prices, very large moves and series stuck at the same value. They are flagged for review, never deleted automatically.
def zero_runs(s, min_len):
    """Runs of at least min_len consecutive zero returns: (start date, length)."""
    z = s.eq(0)
    block = (z != z.shift()).cumsum()
    return [(grp.index[0], len(grp)) for _, grp in s[z].groupby(block[z]) if len(grp) >= min_len]

EXTREME = 0.15
STALE_LEN = 5

for t in ri_clean.columns:
    nonpos = ri_clean.index[(ri_clean[t] <= 0) | (price_clean[t] <= 0)]
    add_check("Non-positive price or RI", t, nonpos, len(nonpos),
              justification="No case" if len(nonpos) == 0 else "Flagged for review")

    ext = r_ri[t][r_ri[t].abs() > EXTREME]
    add_check("Daily return > 15%", t, ext.index, len(ext),
              justification="No case" if len(ext) == 0 else "Flagged for review; no automatic exclusion")

    runs = zero_runs(r_ri[t], STALE_LEN)
    add_check(f"Stale series ({STALE_LEN}+ days)", t, [d for d, _ in runs], len(runs),
              justification="No case" if not runs else "Flagged for review; no automatic exclusion")

# 4. Review of extreme returns: each move above 15% is shown next to SPY's move that day and the next day; after review they match known crises (2008-2009, March 2020) and are kept.
flags = []
for t in ri_clean.columns:
    for d, v in r_ri[t][r_ri[t].abs() > EXTREME].items():
        flags.append({
            "Date": d.date(),
            "Ticker": t,
            "RI return": f"{v:+.1%}",
            "Price return": f"{r_p.loc[d, t]:+.1%}",
            "SPY same day": f"{r_ri.loc[d, 'SPY']:+.1%}",
            "RI return next day": f"{r_ri[t].shift(-1).loc[d]:+.1%}",
        })
if flags:
    display(pd.DataFrame(flags).sort_values("Date"))

# Decision after review: market moves consistent with known crises, kept
report_df = pd.DataFrame(report)
mask = (report_df["Check"] == "Daily return > 15%") & (report_df["Detected"] > 0)
report_df.loc[mask, "Kept"] = report_df.loc[mask, "Detected"]
report_df.loc[mask, "Justification"] = (
    "Kept: market moves consistent with the 2008-2009 and 2020 crises; "
    "Price and RI returns agree at displayed precision; "
    "SPY moves in the same direction; no obvious data error"
)
report = report_df.to_dict("records")

display(report_df[["Check", "Ticker", "Detected", "Kept", "Excluded", "Justification"]])

NYSE sessions: 4992
Non-NYSE dates in file: 186 (exact copies of previous day: 185)
Missing NYSE sessions: 0
After filtering: 4992 sessions, 2007-01-03 to 2026-11-04


,Date,Ticker,RI return,Price return,SPY same day,RI return next day
0,2008-10-13,EFA,+15.9%,+15.9%,+14.5%,-2.1%
1,2008-10-13,EEM,+22.8%,+22.8%,+14.5%,-5.0%
2,2008-10-15,EEM,-16.2%,-16.2%,-9.8%,+4.9%
3,2008-10-28,EEM,+25.6%,+25.6%,+11.7%,-6.8%
4,2008-10-28,VNQ,+15.6%,+15.6%,+11.7%,-3.5%
5,2008-11-24,VNQ,+17.0%,+17.0%,+6.9%,+1.6%
6,2008-12-01,VNQ,-19.5%,-19.5%,-8.9%,+13.3%
7,2009-03-23,VNQ,+16.2%,+16.2%,+7.2%,-7.7%
8,2020-03-16,VNQ,-17.7%,-17.7%,-10.9%,+5.8%


,Check,Ticker,Detected,Kept,Excluded,Justification
0,Non-NYSE date,All,186,0,186,NYSE closed; 185/186 dates match the previous ...
1,Missing NYSE session,All,0,0,0,No case
2,Non-positive price or RI,SPY,0,0,0,No case
3,Daily return > 15%,SPY,0,0,0,No case
4,Stale series (5+ days),SPY,0,0,0,No case
5,Non-positive price or RI,EFA,0,0,0,No case
6,Daily return > 15%,EFA,1,1,0,Kept: market moves consistent with the 2008-20...
7,Stale series (5+ days),EFA,0,0,0,No case
8,Non-positive price or RI,EEM,0,0,0,No case
9,Daily return > 15%,EEM,3,3,0,Kept: market moves consistent with the 2008-20...


## 3. Returns and risk-free rate

Returns come from the total return index (dividends included). A daily return is *regular* when its two dates are consecutive sessions; only regular returns enter the covariance. Monthly returns use the last NYSE session of each month for every ETF.

In [ ]:
gap = (r_ri - r_p).dropna()   # daily gap: RI return minus price return

# Annual performance gap RI - price, full years only
years = r_ri.index.year
annual_ri = (1 + r_ri).groupby(years).prod() - 1
annual_p = (1 + r_p).groupby(years).prod() - 1
annual_gap = (annual_ri - annual_p).iloc[1:-1]   # drop 2007 (start) and 2026 (incomplete)

summary = pd.DataFrame({
    "Mean annual gap RI - Price": annual_gap.mean().map("{:.2%}".format),
    "Days with gap > +0.1%": (gap > 0.001).sum(),
    "Days with gap < -0.1%": (gap < -0.001).sum(),
    "Largest daily gap": gap.abs().max().map("{:.3%}".format),
})
display(summary)

for t in ri_clean.columns:
    neg = gap.index[gap[t] < -0.001]
    add_check("Negative RI - Price gap (> 0.1 pp)", t, neg, len(neg),
              justification="No case" if len(neg) == 0 else "Flagged for review")


# Daily returns with a regularity flag: a return is 'regular' when its two dates are consecutive trading days; only regular returns enter the covariance.
# A return is regular if the two dates are consecutive NYSE sessions
pos = sessions.get_indexer(r_ri.index)
regular = pd.Series(pos, index=r_ri.index).diff().eq(1)

daily_returns = r_ri.copy()
daily_returns["regular"] = regular
daily_returns = daily_returns.iloc[1:]   # first date has no previous observation

print(f"Daily returns: {len(daily_returns)}, irregular: {(~daily_returns['regular']).sum()}")

# Monthly returns on a common NYSE month-end: all ETFs are measured on the same last trading day of each month, so their returns line up.
last_data_date = ri_clean.index.max()
last_month_end = last_data_date + pd.offsets.MonthEnd(0)
full_sessions = pd.DatetimeIndex(
    nyse.schedule(start_date=ri_clean.index.min(), end_date=last_month_end).index
).normalize()
month_ends = full_sessions.to_series().groupby(full_sessions.to_period("M")).max()

unavailable = month_ends[~month_ends.isin(ri_clean.index)]
incomplete_final = unavailable[unavailable > last_data_date]    # month not finished yet
missing_month_end = unavailable[unavailable <= last_data_date]  # genuine data gap

print(f"Month-ends: {len(month_ends)}")
print("Incomplete final month:", [str(p) for p in incomplete_final.index])
print("Missing month-ends (data gap):", [str(p) for p in missing_month_end.index])

# Keep the full monthly calendar: a missing month-end stays visible as NaN
ri_month_end = ri_clean.reindex(month_ends.values)
ri_month_end.index = month_ends.index

# A missing month-end makes this month's and next month's returns NaN
monthly_returns = ri_month_end.pct_change(fill_method=None)
monthly_returns = monthly_returns.iloc[1:]                         # first month: no previous month-end
monthly_returns = monthly_returns.drop(index=incomplete_final.index)

n_valid = monthly_returns.dropna().shape[0]
print(f"Monthly returns: {len(monthly_returns)} months ({n_valid} available), "
      f"from {monthly_returns.index.min()} to {monthly_returns.index.max()}")

add_check("Incomplete final month", "All", list(incomplete_final), len(incomplete_final),
          excluded=len(incomplete_final),
          justification="No case" if len(incomplete_final) == 0
          else "Data end before month-end; monthly return not computed")
add_check("Missing month-end", "All", list(missing_month_end), len(missing_month_end),
          excluded=len(missing_month_end),
          justification="No case" if len(missing_month_end) == 0
          else "Month-end absent from data; returns of this month and the next set to NaN")

# Consistency test: compounding the daily returns of a month must give the monthly return
# (checked on fully covered months only).
sessions_per_month = full_sessions.to_series().groupby(full_sessions.to_period("M")).size()
data_per_month = (ri_clean.index.to_series().groupby(ri_clean.index.to_period("M")).size()
                  .reindex(sessions_per_month.index))
full_months = sessions_per_month.index[sessions_per_month.eq(data_per_month)]

compounded = (1 + r_ri.iloc[1:]).groupby(r_ri.index[1:].to_period("M")).prod() - 1
common = (monthly_returns.dropna().index
          .intersection(compounded.index)
          .intersection(full_months))
max_diff = (monthly_returns.loc[common] - compounded.loc[common]).abs().max().max()

print(f"Months tested: {len(common)}, max difference: {max_diff:.2e}")
assert max_diff < 1e-10, "Monthly returns inconsistent with daily returns"

# Risk-free rate audit (DGS1MO, FRED): check the series as downloaded; it is converted to a
# daily rate later, in src/metrics.py.
rf = pd.read_csv(RF_FILE)
rf.columns = ["Date", "DGS1MO"]
rf["Date"] = pd.to_datetime(rf["Date"], errors="coerce")
rf["DGS1MO"] = pd.to_numeric(rf["DGS1MO"], errors="coerce")   # blanks become NaN
rf = rf.dropna(subset=["Date"]).set_index("Date").sort_index()["DGS1MO"]

print(f"DGS1MO: {len(rf)} dates, {rf.index.min().date()} to {rf.index.max().date()}")
print(f"Duplicated dates: {rf.index.duplicated().sum()}")
print(f"Missing values: {rf.isna().sum()}")

# Missing values on NYSE sessions (e.g. bond market holidays such as Columbus Day)
rf_on_sessions = rf.reindex(sessions)
missing_on_sessions = rf_on_sessions.index[rf_on_sessions.isna()]
print(f"NYSE sessions without a published rate: {len(missing_on_sessions)}")
print("Examples:", [str(d.date()) for d in missing_on_sessions[:8]])

# Longest gap between two published rates on NYSE sessions (staleness)
published = rf_on_sessions.notna()
gap_len = (~published).groupby(published.cumsum()).sum().max()
print(f"Longest run of NYSE sessions without a published rate: {gap_len}")

print(f"Range: {rf.min():.2f}% to {rf.max():.2f}%  (negative values: {(rf < 0).sum()})")

add_check("Risk-free rate missing on NYSE session", "DGS1MO", missing_on_sessions,
          len(missing_on_sessions),
          justification="To be filled with last published rate (V1.4 rule); transformation postponed")

,Mean annual gap RI - Price,Days with gap > +0.1%,Days with gap < -0.1%,Largest daily gap
SPY,2.06%,79,0,0.806%
EFA,3.14%,39,0,2.496%
EEM,2.20%,38,0,2.091%
IEF,2.47%,216,0,0.835%
LQD,3.98%,238,0,0.554%
TLT,3.11%,238,0,0.440%
VNQ,4.32%,79,0,2.647%
GLD,0.00%,0,0,0.007%


Daily returns: 4991, irregular: 0
Month-ends: 239
Incomplete final month: ['2026-11']
Missing month-ends (data gap): []
Monthly returns: 237 months (237 available), from 2007-02 to 2026-10
Months tested: 237, max difference: 1.33e-15
DGS1MO: 5176 dates, 2007-01-02 to 2026-11-03
Duplicated dates: 0
Missing values: 212
NYSE sessions without a published rate: 38
Examples: ['2007-10-08', '2007-11-12', '2008-10-13', '2008-11-11', '2009-10-12', '2009-11-11', '2010-10-11', '2010-11-11']
Longest run of NYSE sessions without a published rate: 1
Range: 0.00% to 6.02%  (negative values: 0)


## 4. Export

Clean prices, daily and monthly returns and the quality report, read by the rest of the project.

In [7]:
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

# 1. Clean daily prices and total return indices (NYSE sessions only)
daily_clean = pd.concat(
    [price_clean.add_prefix("P_"), ri_clean.add_prefix("RI_")], axis=1
)
daily_clean.index.name = "Date"
daily_clean.to_csv(PROCESSED / "etf_daily_clean.csv")

# 2. Daily returns from RI, with regularity flag
daily_returns.index.name = "Date"
daily_returns.to_csv(PROCESSED / "etf_daily_returns.csv")

# 3. Monthly returns, with the month-end date used for each month
monthly_out = monthly_returns.copy()
monthly_out.insert(0, "MonthEnd", month_ends.loc[monthly_out.index].dt.date.values)
monthly_out.index = monthly_out.index.astype(str)
monthly_out.index.name = "Month"
monthly_out.to_csv(PROCESSED / "etf_monthly_returns.csv")

# 4. Data quality report
report_df = pd.DataFrame(report)
report_df.to_csv(REPORTS / "data_quality_report.csv", index=False)

for f in [PROCESSED / "etf_daily_clean.csv", PROCESSED / "etf_daily_returns.csv",
          PROCESSED / "etf_monthly_returns.csv", REPORTS / "data_quality_report.csv"]:
    print(f"{f.name}: {pd.read_csv(f).shape[0]} rows")

etf_daily_clean.csv: 4992 rows
etf_daily_returns.csv: 4991 rows
etf_monthly_returns.csv: 237 rows
data_quality_report.csv: 37 rows


## 5. Check

`src/data.py` must rebuild the same returns from the raw file.

In [8]:
# src/data.py must rebuild exactly what this notebook produced
out = data.build_clean_dataset()

print("Sessions kept:     ", len(out["ri"]))
print("Excluded dates:    ", len(out["excluded_dates"]))
print("Daily returns:     ", len(out["daily_returns"]))
print("Irregular returns: ", int((~out["daily_returns"]["regular"]).sum()))
print("Monthly returns:   ", len(out["monthly_returns"]))
print("Missing month-ends:", len(out["missing_month_end"]))
print("Incomplete final:  ", list(out["incomplete_final"].index.astype(str)))

# Compare with the files exported above
clean, daily, monthly = data.load_processed()
cols = data.TICKERS
print("Max diff daily:  ", float((out["daily_returns"][cols] - daily[cols]).abs().max().max()))
print("Max diff monthly:", float((out["monthly_returns"][cols] - monthly[cols]).abs().max().max()))

Sessions kept:      4992
Excluded dates:     0
Daily returns:      4991
Irregular returns:  0
Monthly returns:    237
Missing month-ends: 0
Incomplete final:   ['2026-11']
Max diff daily:   9.996344030316351e-17
Max diff monthly: 9.974659986866641e-17


## 6. Signals

12-1 momentum (last year's return without the last month) and SPY against its 200-day average, computed by `src/signals.py` at each month-end with data known at that close.

In [9]:
# Signals: 12-1 momentum and SPY trend, computed by src/signals.py (tested in tests/test_signals.py)

import numpy as np

from src import signals

out = data.build_clean_dataset()
monthly = out["monthly_returns"]
month_ends = out["month_ends"]

# Momentum at month-end t uses RI up to month-end t-1 only (month t is skipped): no look-ahead
momentum = signals.momentum_12_1(out["ri"], month_ends, out["incomplete_final"])

# Consistency check: same value by compounding monthly returns t-11 ... t-1
check = (1 + monthly).shift(1).rolling(11, min_periods=11).apply(np.prod, raw=True) - 1
common = momentum.dropna().index.intersection(check.dropna().index)
max_diff = float((momentum.loc[common] - check.loc[common]).abs().max().max())

assert len(common) > 0, "No valid months for momentum comparison"
assert max_diff < 1e-10, "Momentum calculation inconsistent"

# Timing convention: momentum[t] is observed at the close of month t
# (uses RI up to month-end t-1, month t is skipped); weights decided at t
# are traded on the next session and earn the returns of month t+1.

print("First available signal:", momentum.dropna().index[0])
print("Last available signal: ", momentum.dropna().index[-1])
print("Months with a signal:  ", len(momentum.dropna()))
print("Max diff vs compounding:", max_diff)
print("\nLatest 12-1 momentum (%):")
print((momentum.iloc[-1] * 100).round(2).sort_values(ascending=False))

# Export the 12-1 momentum signal (decimal returns, one row per decision month-end)
export = momentum.dropna(how="all").copy()
export.index = export.index.astype(str)
export.index.name = "Month"
export.insert(0, "MonthEnd", month_ends.loc[momentum.dropna(how="all").index].dt.strftime("%Y-%m-%d").values)

path = data.PROCESSED_DIR / "momentum_12_1.csv"
export.to_csv(path)

print("Saved:", path.name, export.shape)
print(export.head(3))

# Trend signal: SPY price relative to its 200-session moving average, at each NYSE month-end
spy = out["price"]["SPY"]
trend = signals.trend_spy_ma200(spy, month_ends, out["incomplete_final"])

# Check: MA200 recomputed by hand on the last month-end uses only the 200 sessions up to that date
last = pd.Timestamp(trend["MonthEnd"].iloc[-1])
manual = spy.loc[:last].tail(200).mean()
assert abs(manual - trend["MA200"].iloc[-1]) < 1e-10, "MA200 inconsistent"

# Timing convention: trend[t] uses prices up to the close of month t;
# weights decided at t are traded on the next session and earn the returns of month t+1.

print("First available signal:", trend.index[0])
print("Last available signal: ", trend.index[-1])
print("Months with a signal:  ", len(trend))
print("Months above MA200:    ", int((trend["Signal"] > 0).sum()))
print("Months below MA200:    ", int((trend["Signal"] < 0).sum()))
print("Sign changes:          ", int((trend["Signal"].diff().abs() > 0).sum()))
print("\nLast 6 months:")
print(trend.tail(6).round(4))

# Export the trend signal
path = data.PROCESSED_DIR / "trend_spy_ma200.csv"
trend.to_csv(path, index_label="Month")
print("Saved:", path.name, trend.shape)

First available signal: 2008-01
Last available signal:  2026-10
Months with a signal:   226
Max diff vs compounding: 8.881784197001252e-16

Latest 12-1 momentum (%):
EEM    23.13
EFA    13.71
SPY    13.02
VNQ     4.32
GLD     3.46
LQD    -4.17
IEF    -4.42
TLT   -10.21
Name: 2026-10, dtype: float64
Saved: momentum_12_1.csv (226, 9)
           MonthEnd       SPY       EFA       EEM       IEF       LQD  \
Month                                                                   
2008-01  2008-01-31  0.035764  0.084301  0.331531  0.108126  0.036341   
2008-02  2008-02-29 -0.007370  0.000000  0.263005  0.122341  0.042280   
2008-03  2008-03-31 -0.044121 -0.037198  0.213586  0.137765  0.041724   

              TLT       VNQ       GLD  
Month                                  
2008-01  0.114022 -0.232376  0.271906  
2008-02  0.100044 -0.216328  0.374833  
2008-03  0.113909 -0.226164  0.463039  
First available signal: 2007-10
Last available signal:  2026-10
Months with a signal:   229
Months a

## 7. Black-Litterman views

Each month: three momentum views (best beats worst ETF in each asset class) and one trend view (equities vs bonds). The size of the views (K) is calibrated on 2009–2015 only.

In [10]:
# Black-Litterman views: 3 momentum views and 1 trend view per month, built by src/signals.py.
# K (view size) is frozen in src/config.py at the 2009-2015 calibration rounded to 2 decimals (checked below).
from src import config

ASSET_CLASSES = config.ASSET_CLASSES
K_MOM, K_TREND, CAP = config.K_MOM, config.K_TREND, config.VIEW_CAP

views = signals.build_views(momentum, trend)
signals.check_views(views)   # relative views, 4 views every month, trend baskets +1 / -1, |Q| <= cap

is_mom = views["View"].str.startswith("Momentum")
trend_rows = views["View"] == "Trend: equities vs bonds"
print("Months with views:", views["Month"].nunique())

# Diagnostics on the calibration window only (2009-2015): no use of the test period
calib = views["Month"].between(pd.Period(config.CALIB_START, "M"), pd.Period(config.CALIB_END, "M"))
summary = pd.DataFrame({
    "Momentum views": views.loc[calib & is_mom, "Q"].abs().describe(percentiles=[0.5, 0.9]),
    "Trend view": views.loc[calib & trend_rows, "Q"].abs().describe(percentiles=[0.5, 0.9]),
}).T[["count", "mean", "50%", "90%", "max"]]
summary["capped share"] = [
    (views.loc[calib & is_mom, "Q"] >= CAP - 1e-12).mean(),
    (views.loc[calib & trend_rows, "Q"].abs() >= CAP - 1e-12).mean(),
]
print("\n|Q| on 2009-2015 (annual, decimal):")
print(summary.round(4))

print("\nViews for the last month:")
last = views[views["Month"] == views["Month"].max()]
print(last[["View", *data.TICKERS, "Q"]].round(4).to_string(index=False))

# Calibration: K such that the 5% cap binds in about 10% of cases on 2009-2015 (calibration window only).
# One K for all momentum views (pooled across asset classes), one K for the trend view.
K_MOM_CAL, K_TREND_CAL = signals.calibrate_k(momentum, trend)
calib_months = signals.calibration_months(momentum.index)

print("Momentum spread, 90th percentile:", round(CAP / K_MOM_CAL, 4))
print("Trend |SPY/MA200 - 1|, 90th pct: ", round(CAP / K_TREND_CAL, 4))
print("\nCalibrated K_MOM:  ", round(K_MOM_CAL, 3))
print("Calibrated K_TREND:", round(K_TREND_CAL, 3))
assert (round(K_MOM_CAL, 2), round(K_TREND_CAL, 2)) == (K_MOM, K_TREND), "config K no longer matches calibration"

# Capped share per asset class with the calibrated K (2009-2015)
per_class = {
    cls: (K_MOM_CAL * pd.Series([momentum.loc[m, members].max() - momentum.loc[m, members].min()
                                 for m in calib_months]) >= CAP).mean()
    for cls, members in ASSET_CLASSES.items()
}
print("\nCapped share per class with calibrated K_MOM:")
print(pd.Series(per_class).round(3))

# Export the Black-Litterman views (P rows and Q, one set of 4 views per decision month)
export = views.copy()
export["Month"] = export["Month"].astype(str)
path = data.PROCESSED_DIR / "bl_views.csv"
export.to_csv(path, index=False)
print("Saved:", path.name, export.shape)

Months with views: 226

|Q| on 2009-2015 (annual, decimal):
                count    mean     50%     90%   max  capped share
Momentum views  252.0  0.0244  0.0224  0.0500  0.05        0.1032
Trend view       84.0  0.0250  0.0231  0.0496  0.05        0.1071

Views for the last month:
                            View  SPY  EFA  EEM     IEF     LQD     TLT  VNQ  GLD      Q
    Momentum Equities: EEM > SPY -1.0 0.00 1.00  0.0000  0.0000  0.0000  0.0  0.0 0.0162
       Momentum Bonds: LQD > TLT  0.0 0.00 0.00  0.0000  1.0000 -1.0000  0.0  0.0 0.0097
Momentum Alternatives: VNQ > GLD  0.0 0.00 0.00  0.0000  0.0000  0.0000  1.0 -1.0 0.0014
        Trend: equities vs bonds  0.6 0.24 0.16 -0.4286 -0.2857 -0.2857  0.0  0.0 0.0144
Momentum spread, 90th percentile: 0.3129
Trend |SPY/MA200 - 1|, 90th pct:  0.1285

Calibrated K_MOM:   0.16
Calibrated K_TREND: 0.389

Capped share per class with calibrated K_MOM:
Equities        0.071
Bonds           0.012
Alternatives    0.226
dtype: float64
Saved: b